In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import erf
from skimage import io, filters, measure
from skimage.color import rgb2gray
from scipy.ndimage import gaussian_filter

PROJECT_ROOT = Path.cwd().parent
print('PROJECT_ROOT:', PROJECT_ROOT)

## Define test image

In [ ]:
image_path = PROJECT_ROOT / Path('tmp/test_images/NP_19_Ho-240629.tif')
print('image_path:', image_path)

## Preprocess image

In [ ]:
def load_sem(filepath: Path) -> np.ndarray:
    # Manage errors: fileNotFoundError + file is not a .tif
    
    # Load Image
    img = io.imread(filepath)

    # Handle RGB (some SEMs save simple TIFs as RGB)
    if img.ndim == 3:
        img = rgb2gray(img)  # normalizes to [0, 1] float
    else:
        # Convert grayscale to float [0.0, 1.0] for consistency
        img = img.astype(np.float64) / 255.0
        
    return img

def clean_sem(img: np.ndarray) -> np.ndarray:
    # Detect the white band
    # Calculate row means from bottom up
    row_means = np.mean(img, axis=1)

    # Calculate the difference between adjacent row means to find sharp changes
    row_gradient = np.diff(row_means)
    
    # Find the largest jump in brightness, which corresponds to the top of the data band.
    # We search in the bottom 20% of the image to avoid finding the NP itself.
    search_start_index = int(img.shape[0] * 0.80)
    crop_point = search_start_index + np.argmax(row_gradient[search_start_index:])
    
    # Crop the image to remove the band (with a small safety margin of 2 pixels)
    cropped_img = img[:crop_point - 2, :]

    return cropped_img

In [ ]:
# Main
img = load_sem(image_path)
no_bar_img = clean_sem(img)

In [ ]:
# Visualize
fig, ax = plt.subplots(1, 2)

ax[0].imshow(img, cmap='gray')
ax[0].set_title("Original Image")

ax[1].imshow(no_bar_img, cmap='gray')
ax[1].set_title("Cleaned Image")

plt.show()

In [ ]:
def find_and_crop_around_largest_region(image, margin=100):
    """


    Assumes particle is the largest region post-thresholding.
    """
    h, w = image.shape
    
    # 0. Preprocess image for better detection: Gausian smoothing
    smooth_img = gaussian_filter(image, sigma=1)

    # 1. Detection of particles via thresholding
    thresh_val = filters.threshold_otsu(smooth_img)
    binary = smooth_img > thresh_val
    labeled = measure.label(binary)
    regions = measure.regionprops(labeled, intensity_image=smooth_img)
    
    if not regions:
        raise ValueError("No particles detected.")
    else:
        print(f'Number of regions detected: {np.size(regions)}.')

    # 2. Find the NP region: the largest region
    best_region = max(regions, key=lambda r: r.area)

    # 3. Define ROI (Crop Box)
    min_row, min_col, max_row, max_col = best_region.bbox
    r_start = int(max(0, min_row - margin))
    r_end   = int(min(h, max_row + margin))
    c_start = int(max(0, min_col - margin))
    c_end   = int(min(w, max_col + margin))
    
    print(f"Cropping to rows: {r_start} to {r_end}, cols: {c_start} to {c_end}")

    img_roi = image[r_start:r_end, c_start:c_end]

    return binary, img_roi

In [ ]:
binary, cropped_img = find_and_crop_around_largest_region(no_bar_img, 100)

# Force no interpolation
# plt.imshow(binary, cmap='binary_r', interpolation='nearest')
# plt.colorbar()
# plt.title(f"Unique values: {np.unique(binary)}")

plt.imshow(cropped_img, cmap='gray')
plt.colorbar()
